In [3]:
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")                      
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

CSV_FILE = "telangana_clean.csv"

TEST_FROM = 2018         
TUNE_UNTIL = 2015         
LEAF_OPTIONS = [1, 2, 3, 5, 8]
RANGE_LOW, RANGE_HIGH = 0.10, 0.90        

CATS = ["District", "Crop", "Season", "Rain_Type"]      
NUMS = ["log_area"]                                     
RAIN_TYPES = ["dry", "normal", "wet"]
FEATURES = CATS + NUMS

df = pd.read_csv(CSV_FILE)
print("Data:", df.shape, " years", df["Crop_Year"].min(), "-", df["Crop_Year"].max())
df["log_area"] = np.log1p(df["Area"])

def add_app_area(data, source):
    """Give each row the area the app would look up: the last known area in `source` (the past).
    If that district + crop + season never appeared, use the crop's median area."""
    last = source.sort_values("Crop_Year").groupby(["District", "Crop", "Season"])["Area"].last()
    crop_med = source.groupby("Crop")["Area"].median()
    all_med = source["Area"].median()
    areas = [last.get((d, c, s), crop_med.get(c, all_med))
             for d, c, s in zip(data["District"], data["Crop"], data["Season"])]
    data = data.copy()
    data["log_area"] = np.log1p(areas)
    return data


def make_model(min_samples_leaf):
    """Pipeline = turn text columns into 0/1 columns, then run the Random Forest."""
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATS),
                              ("num", "passthrough", NUMS)])
    forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=min_samples_leaf,
                                   n_jobs=-1, random_state=42)
    return Pipeline([("prep", prep), ("model", forest)])


def predict_yield(pipe, data):
    """The model predicts log(1 + yield); convert back to tonnes per hectare."""
    return np.expm1(pipe.predict(data[FEATURES]))


def evaluate(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    pct = np.median(np.abs(predicted - actual) / actual) * 100
    return round(float(mae), 3), round(float(rmse), 3), round(float(r2), 3), round(float(pct), 1)


df["rel_yield"] = df["Yield"] / df.groupby(["Crop", "District", "Season"])["Yield"].transform("mean")
df["rel_yield"] = df["rel_yield"].clip(upper=3)

weather = {
    "rain_mm":     ("Rainfall (mm)",           100, "per +100 mm"),
    "temp_max":    ("Avg max temperature (C)", 1,   "per +1 C"),
    "hot_days_35": ("Days above 35C",          10,  "per +10 days"),
}
seasons = ["kharif", "rabi"]

slope_rows = []
fig, axes = plt.subplots(len(seasons), len(weather), figsize=(15, 8))
for r, season in enumerate(seasons):
    s = df[df["Season"] == season]
    for c, (col, (label, step, step_text)) in enumerate(weather.items()):
        ax = axes[r][c]

        # slope of the straight line  rel_yield = a + b * weather
        b, a = np.polyfit(s[col], s["rel_yield"], 1)
        corr = np.corrcoef(s[col], s["rel_yield"])[0, 1]
        change_pct = b * step * 100          # % change in yield for one "step" of weather
        slope_rows.append({"season": season, "weather": label, "step": step_text,
                           "yield_change_pct": round(float(change_pct), 2),
                           "correlation": round(float(corr), 3), "rows": len(s)})

        # 5 groups from low to high weather: dots = average yield of the group
        groups = pd.qcut(s[col], 5, duplicates="drop")
        x_mid = s.groupby(groups, observed=True)[col].mean()
        y_avg = s.groupby(groups, observed=True)["rel_yield"].mean()
        ax.plot(x_mid.values, y_avg.values, marker="o", label="group average")
        xs = np.linspace(s[col].min(), s[col].max(), 50)
        ax.plot(xs, a + b * xs, color="red", linestyle="--", label="slope line")
        ax.axhline(1, color="grey", linestyle=":")
        ax.set_title(f"{season}: {label}\n{change_pct:+.1f}% yield {step_text} (r = {corr:+.2f})", fontsize=10)
        ax.set_xlabel(label)
        ax.set_ylabel("yield vs normal")
        if r == 0 and c == 0:
            ax.legend()
plt.tight_layout()
plt.savefig("weather_effect.png", dpi=120)
plt.close()

slopes = pd.DataFrame(slope_rows)
slopes.to_csv("weather_slopes.csv", index=False)
print("\nWeather slopes (change in yield for a step of weather; r close to 0 = weak link):")
print(slopes.to_string(index=False))

print("\nAverage yield vs normal, by rain type:")
print(df.groupby(["Season", "Rain_Type"])["rel_yield"].mean().round(3).unstack().to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for crop in df["Crop"].value_counts().head(4).index:
    by_year = df[df["Crop"] == crop].groupby("Crop_Year")["Yield"].mean()
    axes[0].plot(by_year.index, by_year.values, marker="o", markersize=3, label=crop)
axes[0].set_title("Average yield by year (4 most common crops)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Yield (t/ha)")
axes[0].legend()

for season in seasons:
    by_year = df[df["Season"] == season].groupby("Crop_Year")["rel_yield"].mean()
    axes[1].plot(by_year.index, by_year.values, marker="o", markersize=3, label=season)
axes[1].axhline(1, color="grey", linestyle=":")
axes[1].axvline(TEST_FROM - 0.5, color="red", linestyle="--", label="test years start")
axes[1].set_title("Average yield vs normal by year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("yield vs normal")
axes[1].legend()
plt.tight_layout()
plt.savefig("yield_by_year.png", dpi=120)
plt.close()

train = df[df["Crop_Year"] < TEST_FROM]
test = add_app_area(df[df["Crop_Year"] >= TEST_FROM], train)
print(f"\nTrain rows: {len(train)} (up to {TEST_FROM - 1})   Test rows: {len(test)} ({TEST_FROM}+)")

fit_part = train[train["Crop_Year"] <= TUNE_UNTIL]
check_part = add_app_area(train[train["Crop_Year"] > TUNE_UNTIL], fit_part)

print("\nTuning min_samples_leaf (checked on 2016-2017):")
best_mae, best_leaf = float("inf"), LEAF_OPTIONS[0]
for leaf in LEAF_OPTIONS:
    pipe = make_model(leaf).fit(fit_part[FEATURES], np.log1p(fit_part["Yield"]))
    mae = mean_absolute_error(check_part["Yield"], predict_yield(pipe, check_part))
    print(f"  min_samples_leaf={leaf} -> MAE {mae:.3f}")
    if mae < best_mae:
        best_mae, best_leaf = mae, leaf
print("Best min_samples_leaf:", best_leaf)

columns = ["MAE (t/ha)", "RMSE (t/ha)", "R2", "Median % error"]
results = {}
crop_avg = train.groupby("Crop")["Yield"].mean()
results["Baseline: crop average"] = evaluate(test["Yield"].values, test["Crop"].map(crop_avg).values)

best_pipe = make_model(best_leaf)
best_pipe.fit(train[FEATURES], np.log1p(train["Yield"]))
test["predicted"] = predict_yield(best_pipe, test)
test["pct_error"] = np.abs(test["predicted"] - test["Yield"]) / test["Yield"] * 100
results["Random Forest (tuned)"] = evaluate(test["Yield"].values, test["predicted"].values)

comparison = pd.DataFrame(results, index=columns).T
comparison.to_csv("model_comparison.csv")
print(f"\nResults on {TEST_FROM}+ (unseen years):\n", comparison.to_string())

accuracy = {}
for column in ["Crop", "District"]:
    table = test.groupby(column).agg(rows=("pct_error", "size"), median_pct_error=("pct_error", "median"))
    table = table[table["rows"] >= 20].sort_values("median_pct_error", ascending=False).round(1)
    print(f"\nLeast certain {column}s:\n", table.head(5).to_string())
    print(f"Most certain {column}s:\n", table.tail(3).to_string())
    table.to_csv(f"error_by_{column.lower()}.csv")
    accuracy[column] = table.reset_index().to_dict("records")

imp = permutation_importance(best_pipe, test[FEATURES], np.log1p(test["Yield"]),
                             scoring="neg_mean_absolute_error", n_repeats=5, random_state=42)
importance = pd.Series(imp.importances_mean, index=FEATURES).clip(lower=0)
importance = (importance / importance.sum() * 100).round(1).sort_values(ascending=False)
importance.to_csv("input_importance.csv", header=["importance_pct"])
print("\nImportance of each input (%):\n", importance.to_string())

checker = make_model(best_leaf).fit(fit_part[FEATURES], np.log1p(fit_part["Yield"]))
check_part["ratio"] = check_part["Yield"] / predict_yield(checker, check_part)
all_low, all_high = check_part["ratio"].quantile([RANGE_LOW, RANGE_HIGH])

range_table = {}
for crop, group in check_part.groupby("Crop"):
    values = group["ratio"] if len(group) >= 30 else check_part["ratio"]   # few rows -> use all crops together
    low, high = values.quantile([RANGE_LOW, RANGE_HIGH])
    range_table[crop] = {"low": round(float(low), 3), "high": round(float(high), 3)}
for crop in df["Crop"].unique():                      # crops with no rows in the checking years
    range_table.setdefault(crop, {"low": round(float(all_low), 3), "high": round(float(all_high), 3)})

for crop, r in range_table.items():
    spread = (r["high"] - r["low"]) / 2 * 100
    r["level"] = "Low" if spread < 35 else ("Medium" if spread < 55 else "High")
pd.DataFrame(range_table).T.to_csv("prediction_ranges.csv")

low = test["predicted"] * test["Crop"].map({c: r["low"] for c, r in range_table.items()})
high = test["predicted"] * test["Crop"].map({c: r["high"] for c, r in range_table.items()})
inside = ((test["Yield"] >= low) & (test["Yield"] <= high)).mean() * 100
print(f"\nShare of real {TEST_FROM}+ yields inside the likely range: {inside:.0f}% (target 80%)")
print("Uncertainty labels:", pd.Series([r["level"] for r in range_table.values()]).value_counts().to_dict())

final_model = make_model(best_leaf)
final_model.fit(df[FEATURES], np.log1p(df["Yield"]))

last_area = df.sort_values("Crop_Year").groupby(["District", "Crop", "Season"])["Area"].last()
valid = {}                                            # {district: {crop: [seasons]}} for the dropdowns
for d, c, s in df[["District", "Crop", "Season"]].drop_duplicates().itertuples(index=False):
    valid.setdefault(d, {}).setdefault(c, []).append(s)

bundle = {
    "model": final_model,
    "model_name": "Random Forest (tuned)",
    "best_leaf": best_leaf,
    "trained_years": [int(df["Crop_Year"].min()), int(df["Crop_Year"].max())],
    "rain_types": RAIN_TYPES,
    "last_area": {key: float(v) for key, v in last_area.items()},
    "crop_median_area": df.groupby("Crop")["Area"].median().to_dict(),
    "ranges": range_table,
    "valid": valid,
    "input_importance": importance.to_dict(),
    "model_comparison": [dict(Model=name, **dict(zip(columns, values))) for name, values in results.items()],
    "accuracy_by_crop": accuracy["Crop"],
    "accuracy_by_district": accuracy["District"],
    "weather_slopes": slopes.to_dict("records"),
}
joblib.dump(bundle,"ready_t.pkl", compress=9)
print(f"\nModel saved as ready_telangana.pkl")


Data: (8233, 16)  years 1997 - 2019

Weather slopes (change in yield for a step of weather; r close to 0 = weak link):
season                 weather         step  yield_change_pct  correlation  rows
kharif           Rainfall (mm)  per +100 mm              3.03        0.205  4005
kharif Avg max temperature (C)     per +1 C             -1.16       -0.040  4005
kharif          Days above 35C per +10 days              2.62        0.066  4005
  rabi           Rainfall (mm)  per +100 mm              1.59        0.017  3641
  rabi Avg max temperature (C)     per +1 C             -1.51       -0.040  3641
  rabi          Days above 35C per +10 days             -3.38       -0.092  3641

Average yield vs normal, by rain type:
Rain_Type     dry  normal    wet
Season                          
kharif      0.874   1.005  1.086
rabi        0.959   1.002  1.030
whole year  0.994   0.965  1.042

Train rows: 6738 (up to 2017)   Test rows: 1495 (2018+)

Tuning min_samples_leaf (checked on 2016-2017):
  m